# Smoke test: `analytics` hit-counter service

Calls the **deployed** `/hit` endpoint directly (no local dev server exists for
this package) and then reads the resulting hourly bucket straight out of S3 to
confirm the write actually landed. Requires `SCW_ACCESS_KEY`/`SCW_SECRET_KEY` in
`analytics/.env` (copy from `analytics/.env.example`) — only for the S3 read-back,
the `POST /hit` call itself needs no credentials.

In [ ]:
import os
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv

from storage import S3Storage

load_dotenv()  # searches upward — finds ../.env (analytics/.env)

ANALYTICS_URL = "https://analytics.fretchen.eu"

## 1. Valid hits — expect `204`

In [ ]:
for _ in range(2):
    response = requests.post(
        f"{ANALYTICS_URL}/hit",
        json={"site": "fretchen.eu", "path": "/notebook-smoke-test"},
    )
    print(response.status_code)

## 2. Invalid site — expect `400`

Confirms `hit.ts`'s `ALLOWED_SITE` validation is actually live on the deployed function.

In [ ]:
bad_response = requests.post(
    f"{ANALYTICS_URL}/hit",
    json={"site": "evil.com", "path": "/x"},
)
print(bad_response.status_code, bad_response.json())

## 3. Read the write back from S3

Confirms the counter object is really there — and that it's **not** publicly
readable without these credentials (see `analytics/README.md` — counters are
private by design).

In [ ]:
s3 = S3Storage(
    access_key=os.environ["SCW_ACCESS_KEY"],
    secret_key=os.environ["SCW_SECRET_KEY"],
)

hour_key = f"counts/fretchen.eu/{datetime.now(timezone.utc):%Y-%m-%dT%H}.json"
bucket = s3.read(hour_key)
print(hour_key)
print(bucket)

assert bucket is not None, "expected the hour bucket to exist after the POSTs above"
assert bucket["pages"]["/notebook-smoke-test"] == 2